# 🚶 Bangkok Inner-District Walkability — H3 Hexbin Map
**OSMnx 2.1.0 + H3 + PyDeck**

ดึงข้อมูลจริงจาก OpenStreetMap → คำนวณ walkability score ต่อ H3 hex → render ด้วย pydeck

**ตัวแปร 6 ตัว:**
1. Footway/Pedestrian path (ทางเท้า)
2. Restaurant (ร้านอาหาร)
3. Café (คาเฟ่)
4. Convenience Store (ร้านสะดวกซื้อ)
5. Park (สวนสาธารณะ)
6. Transit Stop (ป้ายรถเมล์/สถานีรถไฟฟ้า)

## 1. Install & Import

In [ ]:
!pip install -q osmnx h3 pydeck geopandas shapely

In [ ]:
import osmnx as ox
import h3
import pydeck as pdk
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely.geometry import Point, mapping
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

print(f"osmnx: {ox.__version__}")
print(f"h3:    {h3.__version__}")
print(f"pydeck: {pdk.__version__}")

## 2. กำหนดเขตชั้นในกรุงเทพฯ (~15 เขต)
ใช้ชื่อภาษาอังกฤษตาม OSM boundary

In [ ]:
# เขตชั้นในกรุงเทพฯ — ใช้ชื่อที่ OSM Nominatim รู้จัก
INNER_DISTRICTS = [
    "Phra Nakhon, Bangkok, Thailand",
    "Dusit, Bangkok, Thailand",
    "Pomprap Sattru Phai, Bangkok, Thailand",
    "Samphanthawong, Bangkok, Thailand",
    "Bang Rak, Bangkok, Thailand",
    "Sathon, Bangkok, Thailand",
    "Bang Kho Laem, Bangkok, Thailand",
    "Pathum Wan, Bangkok, Thailand",
    "Ratchathewi, Bangkok, Thailand",
    "Phaya Thai, Bangkok, Thailand",
    "Din Daeng, Bangkok, Thailand",
    "Huai Khwang, Bangkok, Thailand",
    "Watthana, Bangkok, Thailand",
    "Khlong Toei, Bangkok, Thailand",
    "Chatuchak, Bangkok, Thailand",
]

print(f"จำนวนเขต: {len(INNER_DISTRICTS)}")

## 3. ดึง Boundary Polygon รวม

In [ ]:
# ดึง boundary ของแต่ละเขต แล้วรวมเป็น polygon เดียว
district_gdfs = []
for name in INNER_DISTRICTS:
    try:
        gdf = ox.geocode_to_gdf(name)
        district_gdfs.append(gdf)
        print(f"  ✓ {name}")
    except Exception as e:
        print(f"  ✗ {name} — {e}")

districts = pd.concat(district_gdfs, ignore_index=True)
study_area = districts.union_all()  # dissolved polygon
print(f"\nรวม {len(district_gdfs)} เขต → study area polygon")

## 4. ดึงข้อมูล OSM (6 ชั้นข้อมูล)
ใช้ `ox.features.features_from_polygon()` — ดึงจาก Overpass API จริง

In [ ]:
ox.settings.timeout = 300  # เพิ่ม timeout เผื่อ Overpass ช้า

print("📡 กำลังดึงข้อมูลจาก OSM...")
print("   (อาจใช้เวลา 2-5 นาที)\n")

# --- 1) Walking network (footway, pedestrian, path) ---
print("▶ Walking network...")
try:
    G_walk = ox.graph_from_polygon(study_area, network_type='walk')
    edges = ox.graph_to_gdfs(G_walk, nodes=False, edges=True)
    # เอาเฉพาะ footway/pedestrian
    foot_mask = edges['highway'].apply(
        lambda x: any(t in str(x) for t in ['footway', 'pedestrian', 'path', 'steps'])
    )
    footways = edges[foot_mask].copy()
    print(f"  ✓ footway/pedestrian: {len(footways)} segments, {footways['length'].sum()/1000:.1f} km")
except Exception as e:
    footways = gpd.GeoDataFrame()
    print(f"  ✗ Walking network failed: {e}")

# --- 2) Restaurant ---
print("▶ Restaurant...")
try:
    restaurants = ox.features_from_polygon(study_area, tags={'amenity': 'restaurant'})
    print(f"  ✓ restaurant: {len(restaurants)}")
except Exception as e:
    restaurants = gpd.GeoDataFrame()
    print(f"  ✗ {e}")

# --- 3) Café ---
print("▶ Café...")
try:
    cafes = ox.features_from_polygon(study_area, tags={'amenity': 'cafe'})
    print(f"  ✓ café: {len(cafes)}")
except Exception as e:
    cafes = gpd.GeoDataFrame()
    print(f"  ✗ {e}")

# --- 4) Convenience Store ---
print("▶ Convenience Store...")
try:
    convstores = ox.features_from_polygon(study_area, tags={'shop': 'convenience'})
    print(f"  ✓ convenience store: {len(convstores)}")
except Exception as e:
    convstores = gpd.GeoDataFrame()
    print(f"  ✗ {e}")

# --- 5) Park ---
print("▶ Park...")
try:
    parks = ox.features_from_polygon(
        study_area,
        tags={'leisure': 'park'}
    )
    print(f"  ✓ park: {len(parks)}")
except Exception as e:
    parks = gpd.GeoDataFrame()
    print(f"  ✗ {e}")

# --- 6) Transit Stop ---
print("▶ Transit Stop...")
try:
    transit = ox.features_from_polygon(
        study_area,
        tags={
            'highway': 'bus_stop',
            'railway': ['station', 'halt'],
            'station': 'subway',
            'amenity': 'bus_station'
        }
    )
    print(f"  ✓ transit stop: {len(transit)}")
except Exception as e:
    transit = gpd.GeoDataFrame()
    print(f"  ✗ {e}")

print("\n✅ ดึงข้อมูลเสร็จ!")

## 5. สร้าง H3 Grid + คำนวณ Score
แบ่ง study area เป็น H3 resolution 9 (~174m per hex)

In [ ]:
H3_RES = 9  # ~174m edge length — ละเอียดพอเห็น pattern

# แปลง study_area polygon → H3 hex set
coords = list(mapping(study_area)['coordinates'][0])
# h3.polygon_to_cells ต้องการ (lat, lng) ไม่ใช่ (lng, lat)
h3_polygon = [(lat, lng) for lng, lat in coords]

try:
    hex_ids = h3.polygon_to_cells(h3.LatLngPoly(h3_polygon), H3_RES)
except:
    # fallback สำหรับ h3 version เก่า
    hex_ids = h3.polyfill_geojson(mapping(study_area), H3_RES)

hex_ids = list(hex_ids)
print(f"H3 resolution {H3_RES}: {len(hex_ids):,} hexagons")

In [ ]:
def get_centroid_coords(gdf):
    """แปลง GeoDataFrame → list ของ (lat, lng) centroids"""
    if gdf.empty:
        return []
    centroids = gdf.geometry.centroid
    return list(zip(centroids.y, centroids.x))

def count_points_per_hex(coords_list, hex_ids_set, res):
    """นับจำนวน point ที่ตกใน hex แต่ละอัน"""
    counts = defaultdict(int)
    for lat, lng in coords_list:
        try:
            h = h3.latlng_to_cell(lat, lng, res)
        except:
            h = h3.geo_to_h3(lat, lng, res)
        if h in hex_ids_set:
            counts[h] += 1
    return counts

def sum_length_per_hex(line_gdf, hex_ids_set, res, sample_step=50):
    """ประมาณความยาว line ที่ตกใน hex แต่ละอัน (sample points along line)"""
    lengths = defaultdict(float)
    if line_gdf.empty:
        return lengths
    for _, row in line_gdf.iterrows():
        geom = row.geometry
        line_len = geom.length  # degrees → ใช้ row['length'] (meters) ถ้ามี
        seg_len_m = row.get('length', line_len * 111000)  # rough m
        n_samples = max(2, int(seg_len_m / sample_step))
        for i in range(n_samples):
            pt = geom.interpolate(i / n_samples, normalized=True)
            try:
                h = h3.latlng_to_cell(pt.y, pt.x, res)
            except:
                h = h3.geo_to_h3(pt.y, pt.x, res)
            if h in hex_ids_set:
                lengths[h] += seg_len_m / n_samples
    return lengths

print("คำนวณ feature ต่อ hex...")
hex_set = set(hex_ids)

# Footway length per hex (meters)
print("  ▶ footway length...")
foot_len = sum_length_per_hex(footways, hex_set, H3_RES)

# POI counts per hex
print("  ▶ restaurants...")
rest_count = count_points_per_hex(get_centroid_coords(restaurants), hex_set, H3_RES)

print("  ▶ cafés...")
cafe_count = count_points_per_hex(get_centroid_coords(cafes), hex_set, H3_RES)

print("  ▶ convenience stores...")
conv_count = count_points_per_hex(get_centroid_coords(convstores), hex_set, H3_RES)

print("  ▶ parks...")
park_count = count_points_per_hex(get_centroid_coords(parks), hex_set, H3_RES)

print("  ▶ transit stops...")
transit_count = count_points_per_hex(get_centroid_coords(transit), hex_set, H3_RES)

print("✅ เสร็จ!")

In [ ]:
# สร้าง DataFrame
df = pd.DataFrame({'hex': hex_ids})
df['footway_m'] = df['hex'].map(foot_len).fillna(0)
df['restaurant'] = df['hex'].map(rest_count).fillna(0).astype(int)
df['cafe'] = df['hex'].map(cafe_count).fillna(0).astype(int)
df['convenience'] = df['hex'].map(conv_count).fillna(0).astype(int)
df['park'] = df['hex'].map(park_count).fillna(0).astype(int)
df['transit'] = df['hex'].map(transit_count).fillna(0).astype(int)

# Normalize แต่ละตัวแปร (0-1)
features = ['footway_m', 'restaurant', 'cafe', 'convenience', 'park', 'transit']
for col in features:
    col_max = df[col].max()
    df[f'{col}_norm'] = df[col] / col_max if col_max > 0 else 0

# Weighted score (ปรับ weight ได้)
WEIGHTS = {
    'footway_m_norm':    0.25,
    'restaurant_norm':   0.15,
    'cafe_norm':         0.15,
    'convenience_norm':  0.15,
    'park_norm':         0.15,
    'transit_norm':      0.15,
}

df['score'] = sum(df[col] * w for col, w in WEIGHTS.items())
df['score'] = (df['score'] / df['score'].max() * 100).round(1)
df['score'] = df['score'].fillna(0)

# lat/lng ของ hex center (สำหรับ tooltip)
def hex_to_latlng(h):
    try:
        return h3.cell_to_latlng(h)
    except:
        return h3.h3_to_geo(h)

df['lat'] = df['hex'].apply(lambda h: hex_to_latlng(h)[0])
df['lng'] = df['hex'].apply(lambda h: hex_to_latlng(h)[1])

print(f"Score summary:")
print(df['score'].describe())
print(f"\nTop 10 walkable hexagons:")
df.nlargest(10, 'score')[['hex', 'score', 'footway_m', 'restaurant', 'cafe', 'convenience', 'park', 'transit']]

## 6. Render H3 Hexbin Map ด้วย PyDeck 🗺️

In [ ]:
# Plasma colormap
def plasma_rgb(t):
    """t in [0,1] → [R, G, B]"""
    t = max(0, min(1, t))
    c = [
        [13,8,135],[65,4,157],[106,0,168],[143,13,164],
        [177,42,144],[203,71,120],[224,100,97],[240,130,73],
        [251,163,48],[253,196,39],[240,229,33]
    ]
    idx = t * (len(c) - 1)
    i = int(idx)
    f = idx - i
    a = c[min(i, len(c)-1)]
    b = c[min(i+1, len(c)-1)]
    return [int(a[j] + (b[j]-a[j]) * f) for j in range(3)]

df['color'] = df['score'].apply(lambda s: plasma_rgb(s / 100))
df['r'] = df['color'].apply(lambda c: c[0])
df['g'] = df['color'].apply(lambda c: c[1])
df['b'] = df['color'].apply(lambda c: c[2])

layer = pdk.Layer(
    "H3HexagonLayer",
    df,
    pickable=True,
    stroked=True,
    filled=True,
    extruded=True,
    get_hexagon="hex",
    get_fill_color="[r, g, b, 200]",
    get_line_color=[255, 255, 255, 30],
    get_elevation="score",
    elevation_scale=20,
    elevation_range=[0, 100],
    coverage=0.9,
    line_width_min_pixels=0.5,
)

view_state = pdk.ViewState(
    latitude=13.755,
    longitude=100.520,
    zoom=12,
    pitch=45,
    bearing=-15,
)

tooltip = {
    "html": """
    <div style="font-family: sans-serif; padding: 4px;">
        <b>Walkability Score: {score}/100</b><br/>
        🚶 Footway: {footway_m:.0f} m<br/>
        🍜 Restaurant: {restaurant}<br/>
        ☕ Café: {cafe}<br/>
        🏪 Convenience: {convenience}<br/>
        🌳 Park: {park}<br/>
        🚇 Transit: {transit}
    </div>
    """,
    "style": {
        "backgroundColor": "rgba(13,17,23,0.9)",
        "color": "#c9d1d9",
        "fontSize": "12px",
        "borderRadius": "6px",
    },
}

# Basemap ฟรี — ไม่ต้องใช้ Mapbox API key
CARTO_DARK = 'https://basemaps.cartocdn.com/gl/dark-matter-gl-style/style.json'
CARTO_LIGHT = 'https://basemaps.cartocdn.com/gl/positron-gl-style/style.json'

deck = pdk.Deck(
    layers=[layer],
    initial_view_state=view_state,
    tooltip=tooltip,
    map_style=CARTO_DARK,  # เปลี่ยนเป็น CARTO_LIGHT ถ้าอยากได้พื้นขาว
)

deck.to_html('bkk_walkability_h3.html')
print('📄 Saved: bkk_walkability_h3.html')
deck

### 6b. Alternative: Folium + h3 (ถ้า PyDeck มีปัญหา)
Folium ใช้ Leaflet — basemap ฟรี 100% ไม่ต้อง API key เลย

In [ ]:
# ถ้า pydeck render ไม่ออก ใช้ folium แทนได้
# !pip install -q folium h3 branca

import folium
from folium.plugins import FloatImage
import branca.colormap as cm

# สร้าง colormap
colormap = cm.LinearColormap(
    colors=['#0d0887','#5b02a3','#9a179b','#cb4679','#eb7852','#fbb32b','#f0f921'],
    vmin=0, vmax=100,
    caption='Walkability Score'
)

m = folium.Map(
    location=[13.755, 100.520],
    zoom_start=12,
    tiles='CartoDB dark_matter',  # basemap ฟรี
)

# วาด hex ทีละอัน
for _, row in df.iterrows():
    if row['score'] < 1:
        continue  # ข้าม hex ที่ score ≈ 0 เพื่อความเร็ว
    try:
        boundary = h3.cell_to_boundary(row['hex'])
    except:
        boundary = h3.h3_to_geo_boundary(row['hex'])
    polygon = [(lat, lng) for lat, lng in boundary]
    color = colormap(row['score'])
    folium.Polygon(
        locations=polygon,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.6,
        weight=0.5,
        popup=folium.Popup(
            f"Score: {row['score']}/100<br>"
            f"Footway: {row['footway_m']:.0f}m<br>"
            f"Restaurant: {row['restaurant']}<br>"
            f"Café: {row['cafe']}<br>"
            f"Convenience: {row['convenience']}<br>"
            f"Park: {row['park']}<br>"
            f"Transit: {row['transit']}",
            max_width=200
        ),
    ).add_to(m)

colormap.add_to(m)
m.save('bkk_walkability_folium.html')
print('📄 Saved: bkk_walkability_folium.html')
m

## 7. Summary Stats

In [ ]:
# สรุปว่าดึงข้อมูลได้เท่าไหร่
summary = {
    'Footway segments': len(footways),
    'Footway total km': round(footways['length'].sum()/1000, 1) if not footways.empty else 0,
    'Restaurants': len(restaurants),
    'Cafés': len(cafes),
    'Convenience Stores': len(convstores),
    'Parks': len(parks),
    'Transit Stops': len(transit),
    'H3 Hexagons': len(hex_ids),
    'H3 Resolution': H3_RES,
    'Mean Score': round(df['score'].mean(), 1),
    'Max Score': round(df['score'].max(), 1),
}
for k, v in summary.items():
    print(f"  {k}: {v}")

## 8. Export เป็น GeoJSON (optional)
เอาไปเปิดใน QGIS หรือ Kepler.gl ได้เลย

In [ ]:
# สร้าง GeoDataFrame จาก hex boundaries
from shapely.geometry import Polygon

def hex_to_polygon(h):
    try:
        boundary = h3.cell_to_boundary(h)
    except:
        boundary = h3.h3_to_geo_boundary(h)
    # boundary เป็น list ของ (lat, lng) → ต้องแปลงเป็น (lng, lat)
    return Polygon([(lng, lat) for lat, lng in boundary])

df['geometry'] = df['hex'].apply(hex_to_polygon)
gdf_out = gpd.GeoDataFrame(
    df[['hex', 'score', 'footway_m', 'restaurant', 'cafe', 'convenience', 'park', 'transit', 'geometry']],
    geometry='geometry',
    crs='EPSG:4326'
)
gdf_out.to_file('bkk_walkability_h3.geojson', driver='GeoJSON')
print("📄 Saved: bkk_walkability_h3.geojson")